# Verifica output — SIOPE uscite 2025
Certificazione qualità CLEAN e MART. Richiede run locale (out/data/).

In [1]:
import duckdb, pandas as pd
from pathlib import Path
con = duckdb.connect()
ROOT = Path('../../out/data')
A = 2025
CLEAN = ROOT / f'clean/siope_uscite_comuni/{A}/siope_uscite_comuni_{A}_clean.parquet'
MART_PRO = ROOT / f'mart/siope_uscite_comuni/{A}/siope_uscite_comuni_agg_labeled.parquet'
print(f'Verifica uscite {A}')
print(f'Clean: {CLEAN}')

Verifica uscite 2025
Clean: ../../out/data/clean/siope_uscite_comuni/2025/siope_uscite_comuni_2025_clean.parquet


In [2]:
c1 = con.execute(f"""select count(*),count(distinct codice_ente),count(distinct periodo),
  count(*) filter(where importo<0), count(*) filter(where codice_ente is null)
  from read_parquet('{CLEAN}')""").fetchone()
print(f'Righe: {c1[0]:,}\nEnti: {c1[1]:,}\nPeriodi: {c1[2]}\nImporti neg: {c1[3]}\nNull ente: {c1[4]}')

Righe: 8,713,650
Enti: 11,734
Periodi: 12
Importi neg: 2
Null ente: 0


In [3]:
j = con.execute(f"""select
  round(100.*count(*) filter(where provincia is not null)/count(*),2),
  round(100.*count(*) filter(where has_codgest_match)/count(*),2),
  round(100.*count(*) filter(where denominazione_ente is not null)/count(*),2)
  from read_parquet('{CLEAN}')""").fetchone()
print(f'Join territorio: {j[0]}%\nJoin codgest: {j[1]}%\nEnti trovati: {j[2]}%')

Join territorio: 100.0%
Join codgest: 100.0%
Enti trovati: 100.0%


In [4]:
display(con.execute(f"""select codice_comparto,count(*)righe,round(sum(importo)/1e8,2)tot
  from read_parquet('{CLEAN}')
  where codice_comparto is not null group by codice_comparto order by codice_comparto""").fetchdf())

,codice_comparto,righe,tot
0,AAI,5722,552.06
1,ASP,15009,2179.57
2,CDC,42976,1470.96
3,EGP,18946,222.78
4,EPF,36663,152.34
5,FLS,8632,557.22
6,MON,305463,12842.04
7,PRO,7419211,130624.02
8,REG,86920,267337.12
9,RIC,32016,8181.46


In [5]:
m = con.execute(f"""select count(*),round(sum(importo_totale_eur),0),
  count(*) filter(where importo_totale_eur<0),
  count(*) filter(where macro_area is null),
  round(100.*count(*) filter(where macro_categoria='Altre spese')/count(*),2)
  from read_parquet('{MART_PRO}')""").fetchone()
print(f'PRO: {m[0]:,} righe, {m[1]/1e9:.2f} mld, neg {m[2]}, null area {m[3]}, altro {m[4]}%')

PRO: 732,675 righe, 114.20 mld, neg 0, null area 0, altro 8.56%


In [6]:
for s,l in [('regioni','REG'),('sanita','SAN'),('universita','UNI')]:
  path = ROOT / f'mart/siope_uscite_comuni/{A}/siope_uscite_{s}_agg_labeled.parquet'
  r = con.execute(f'select count(*),round(sum(importo_totale_eur)/1e9,2) from read_parquet("{path}")').fetchone()
  print(f'  {l}: {r[0]:>6} righe, {r[1]:>8} mld')

  REG:   8715 righe,   267.34 mld
  SAN:  21619 righe,   177.92 mld
  UNI:  10669 righe,     25.8 mld


In [7]:
display(con.execute(f"""select macro_area,macro_categoria,
  round(sum(importo_totale_eur)/1e9,2)mld from read_parquet('{MART_PRO}')
  where is_titolo_9=false group by all order by mld desc limit 10""").fetchdf())

,macro_area,macro_categoria,mld
0,Spese correnti,Acquisto beni e servizi,37.13
1,Spese in conto capitale,Contributi investimenti,21.89
2,Anticipazioni e partite di giro,Anticipazioni,18.00
3,Spese correnti,Personale,14.20
4,Spese correnti,Trasferimenti correnti,8.34
5,Altre spese,Altre spese,2.86
6,Rimborso prestiti,Rimborso prestiti,2.84
7,Spese correnti,Altre spese,2.14
8,Spese in conto capitale,Trasferimenti c/capitale,2.06
9,Spese correnti,Poste correttive,1.38


In [8]:
checks=[
  ('Clean righe >1M', '✅' if c1[0]>1_000_000 else '❌'),
  ('Clean 12 periodi', '✅' if c1[2]==12 else '❌'),
  ('Importi neg <100', '✅' if c1[3]<100 else '⚠️'),
  ('Join territorio >95%', '✅' if j[0]>95 else '❌'),
  ('Join codgest >95%', '✅' if j[1]>95 else '❌'),
  ('Mart PRO righe >100K', '✅' if m[0]>100_000 else '❌'),
]
display(pd.DataFrame(checks,columns=['check','esito']))
con.close()
print('Verifica completata')

,check,esito
0,Clean righe >1M,✅
1,Clean 12 periodi,✅
2,Importi neg <100,✅
3,Join territorio >95%,✅
4,Join codgest >95%,✅
5,Mart PRO righe >100K,✅


Verifica completata
